# B2 · 奧卡姆剃刀的算術，與 Lindley 悖論

> 上一個 notebook 得到了判決。這一個問兩件更難的事：
> **(1)** 為什麼「擬合更好」的模型會輸？**(2)** 這個判決有多依賴我隨手挑的先驗？

第二個問題是計劃書主題七第三卡明確警告過的弱點，這裡不迴避。

In [1]:
import sys, os, warnings; warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.abspath('../src'))
import numpy as np, json
import data, models, evidence as ev, plots
DATA = os.path.abspath('../../data/B_astro')
RES = json.load(open('../figures/results.json'))

## 1 · 奧卡姆懲罰是「先驗體積」，不是外加的懲罰項

nested sampling 把 $d$ 維積分改寫成對**先驗體積** $X$ 的一維積分：

$$Z = \int_0^1 L(X)\, dX, \qquad X(\lambda) = \int_{L(\theta) > \lambda} p(\theta)\, d\theta$$

用群活點由外往內收縮來估 $X$。奧卡姆因子就**內建在裡面**：參數多、先驗鬆的模型，
高似然區在先驗體積中占的比例小，$Z$ 自然被壓低。**不需要 AIC/BIC 那種外加的懲罰項。**

粗略地說 $Z \approx L_{\max} \times \prod_i (\sigma_{\text{post},i} / \sigma_{\text{prior},i})$，所以

$$\underbrace{\log L_{\max} - \log Z}_{\text{奧卡姆懲罰}} \approx \sum_i \log \frac{\sigma_{\text{prior},i}}{\sigma_{\text{post},i}}$$

拿 Kepler-10b 的實際數字把這筆帳算出來：

In [2]:
E = RES['evidence']['kepler10b']
print(f"{'模型':<5}{'參數':>4}{'max logL':>12}{'log Z':>12}{'奧卡姆懲罰':>12}{'每參數':>9}")
for m, nd in (('M0', 2), ('M1', 8), ('M2', 9)):
    pen = E['maxlogl'][m] - E['logz'][m]
    print(f"{m:<5}{nd:>4}{E['maxlogl'][m]:>12.1f}{E['logz'][m]:>12.1f}"
          f"{pen:>12.1f}{pen/nd:>9.2f}")

dL = E['maxlogl']['M2'] - E['maxlogl']['M1']
dZ = E['logz']['M2'] - E['logz']['M1']
print(f"\nM2 相對 M1：擬合改善 {dL:+.2f} nat，多付奧卡姆代價 {dL-dZ:+.2f} nat"
      f"  →  淨值 {dZ:+.2f} nat")
print(f"換算 log10 B(M2/M1) = {dZ/np.log(10):+.2f} → 資料**不值得**多這一個參數")

模型     參數    max logL       log Z       奧卡姆懲罰      每參數
M0      2      1351.5      1341.7         9.8     4.90
M1      8      1758.3      1727.3        31.0     3.88
M2      9      1761.0      1724.3        36.7     4.08

M2 相對 M1：擬合改善 +2.70 nat，多付奧卡姆代價 +5.68 nat  →  淨值 -2.98 nat
換算 log10 B(M2/M1) = -1.29 → 資料**不值得**多這一個參數


這就是通關標準第 4 條的現場：**M2 擬合得更好（$\log L_{\max}$ 更高），
邊際似然卻更低**。多的那個參數 $J$ 買到的擬合改善，抵不過它拉大的先驗體積。

反過來看 KOI-6017.01——同一個參數在那裡是**划算**的：

In [3]:
F = RES['evidence']['koi6017']
dL2 = F['maxlogl']['M2'] - F['maxlogl']['M1']
dZ2 = F['logz']['M2'] - F['logz']['M1']
print(f"KOI-6017.01  M2 相對 M1：擬合改善 {dL2:+.1f} nat，"
      f"奧卡姆代價 {dL2-dZ2:+.1f} nat → 淨值 {dZ2:+.1f} nat")
print(f"log10 B(M2/M1) = {dZ2/np.log(10):+.2f}  → 決定性支持食雙星")
print(f"\n同一個參數 J：對 Kepler-10b 是浪費，對 KOI-6017.01 值 {dZ2:.0f} nat。")
print("差別不在模型，在資料——這正是邊際似然該有的行為。")

KOI-6017.01  M2 相對 M1：擬合改善 +25.7 nat，奧卡姆代價 +4.1 nat → 淨值 +21.6 nat
log10 B(M2/M1) = +9.39  → 決定性支持食雙星

同一個參數 J：對 Kepler-10b 是浪費，對 KOI-6017.01 值 22 nat。
差別不在模型，在資料——這正是邊際似然該有的行為。


### 那用 BIC 不就好了？

BIC $= k\ln n - 2\ln L_{\max}$ 常被當成邊際似然的廉價替代品——它其實是
$-2\log Z$ 在「拉普拉斯近似 + 單位資訊先驗 + $n \to \infty$」下的漸近展開。
既然是近似，就該問它在**邊緣案例**上準不準：

In [4]:
n = RES['targets']['kepler10b']['nbins']
print(f"（資料點數 n = {n}；BIC 越小越好，log Z 越大越好）\n")
print(f"{'目標':<14}{'ΔBIC(M2−M1)':>14}{'BIC 判':>9}{'ΔlogZ(M2−M1)':>15}{'證據判':>9}   一致？")
for key in ('kepler10b', 'koi6017'):
    E = RES['evidence'][key]
    dBIC = np.log(n) - 2*(E['maxlogl']['M2'] - E['maxlogl']['M1'])
    dlogZ = E['logz']['M2'] - E['logz']['M1']
    v_bic, v_ez = ('M2' if dBIC < 0 else 'M1'), ('M2' if dlogZ > 0 else 'M1')
    print(f"{RES['evidence'][key]['title']:<14}{dBIC:>14.2f}{v_bic:>9}"
          f"{dlogZ:>15.2f}{v_ez:>9}   {'✓' if v_bic == v_ez else '✗ 相反！'}")

（資料點數 n = 164；BIC 越小越好，log Z 越大越好）

目標               ΔBIC(M2−M1)    BIC 判   ΔlogZ(M2−M1)      證據判   一致？
Kepler-10b             -0.30       M2          -2.98       M1   ✗ 相反！
KOI-6017.01           -46.30       M2          21.62       M2   ✓


在證據壓倒性的 KOI-6017.01 上兩者一致；在**差距只有幾 nat 的 Kepler-10b 上，
BIC 和真正的邊際似然可以指向不同的模型**。

這完全符合理論預期——BIC 丟掉了先驗的形狀與寬度，只留 $k\ln n$ 這個粗糙的懲罰。
而本專案的核心恰恰是**先驗承載物理假設**（行星半徑上限 0.2 vs 伴星 1.0），
那正是 BIC 看不見的資訊。**邊緣案例上，該算的積分就得真的算。**

## 2 · 數值可靠性：這些差距真的可信嗎？

Kepler-10b 的 M1 只贏 M2 約 2 nat。這麼小的差距，**必須先證明它不是抽樣噪聲**。

ultranest 回報的 `logzerr` 只涵蓋 nested sampling 的統計誤差，**不涵蓋** step sampler
走不夠遠造成的偏差。所以做了兩件事：

1. **nsteps 階梯檢查**：把 `SliceSampler` 的步數 $2\times \to 4\times \to 8\times$ ndim，看 logZ 是否穩定。
2. **多 seed 重複**：主分析每個 logZ 跑三個獨立 seed，用**實測的 seed 間散布**當誤差棒。

第 1 項發現 $2\times$ 時 logZ 還會擺動約 2.6 nat——**比 Kepler-10b 的 M1/M2 差距還大**。
所以正式設定改用 $4\times$。這個修正是必要的，不是保險。

In [5]:
print(f"nsteps 階梯的 logZ 擺動（2×→8×）= {RES['nsteps_drift']:.2f} nat")
print(f"seed 間散布（三次獨立重複）：")
for key in ('kepler10b', 'koi6017'):
    sp = RES['evidence'][key]['logz_spread']
    print(f"  {key:<12} " + "  ".join(f"{m}={sp[m]:.2f}" for m in ('M0','M1','M2')))
gap = abs(RES['evidence']['kepler10b']['logz']['M2'] - RES['evidence']['kepler10b']['logz']['M1'])
print(f"\n對照：Kepler-10b 的 M1/M2 差距 = {gap:.2f} nat")

nsteps 階梯的 logZ 擺動（2×→8×）= 0.67 nat
seed 間散布（三次獨立重複）：
  kepler10b    M0=0.40  M1=0.77  M2=1.23
  koi6017      M0=0.32  M1=0.25  M2=0.56

對照：Kepler-10b 的 M1/M2 差距 = 2.98 nat


## 3 · 先驗敏感度：貝氏因子**不是**先驗無關的

貝氏因子最常被批評的一點：它對先驗的寬度敏感，而參數估計（後驗）通常不會。
原因就是上面的 Occam 因子——先驗放寬 $k$ 倍，高似然區的占比就降 $k$ 倍：

$$\log_{10} B \;\longrightarrow\; \log_{10} B - \log_{10} k$$

這是**可以定量驗證的預測**，不只是定性警告。把 $R_c/R_*$ 的先驗上界放寬 $k$ 倍重算：

![先驗敏感度](../figures/05_lindley.png)

In [6]:
for s in RES['prior_scan']:
    base = s['log10_B'][s['base_idx']]
    print(f"\n{s['title']}（基準 log10 B = {base:.1f}）")
    print(f"  {'k':>6}{'rp_max':>9}{'log10 B':>12}{'實測 Δ':>10}{'Occam 預測':>12}")
    for rp, b in zip(s['rp_max'], s['log10_B']):
        k = rp / s['rp_max'][s['base_idx']]
        print(f"  {k:>6.1f}{rp:>9.2f}{b:>12.2f}{b-base:>10.2f}{-np.log10(k):>12.2f}")


Kepler-10b（基準 log10 B = 167.4）
       k   rp_max     log10 B      實測 Δ    Occam 預測
     1.0     0.20      167.38      0.00       -0.00
     2.0     0.40      167.12     -0.25       -0.30
     5.0     1.00      166.67     -0.71       -0.70
    10.0     2.00      166.41     -0.97       -1.00

KOI-6017.01（基準 log10 B = 1316.2）
       k   rp_max     log10 B      實測 Δ    Occam 預測
     1.0     0.20     1316.21      0.00       -0.00
     2.0     0.40     1316.67      0.46       -0.30
     5.0     1.00     1316.61      0.40       -0.70
    10.0     2.00     1316.50      0.30       -1.00


**Kepler-10b 完美符合**：觀測點幾乎疊在 $-\log_{10}k$ 的虛線上，
放寬 10 倍掉 0.97，理論預測 1.00——奧卡姆因子被定量證實了。

**KOI-6017.01 卻反過來走。** 這不是數值誤差，是上一個 notebook 那個觀察的直接後果：
**它的 M1 把 $R_c/R_*$ 頂在先驗邊界上**（0.187 vs 上限 0.2）。
Occam 因子的推導假設「放寬先驗只稀釋密度、不改變最大似然」——
而當邊界正**截斷**著最佳解時，放寬它會讓 M1 找到更好的擬合，
似然增益抵消掉體積稀釋，淨效應可以是正的。

這個對比逼出一件重要的事：

> **先驗範圍不是模型的旋鈕，先驗範圍就是模型的定義。**
> 把 M1 的上限從 0.2 放寬到 2.0，你不是「讓行星模型更寬鬆」，
> 而是把它換成了另一個模型（允許恆星尺度伴星的那種）——
> 那已經不是在檢驗「這是不是行星」了。

也必須誠實說清楚：**在這兩個目標上，先驗敏感度不改變任何結論**。
證據強度是 $10^{167}$ 和 $10^{1316}$ 的量級，挪動一個 dex 動不了它。
把這叫「親眼看到 Lindley 悖論」是不老實的——悖論咬人的地方在**弱證據**。

## 4 · 那就造一個弱證據的案例

用 Kepler-10b 真實的每箱誤差當雜訊，注入一個深度可調的合成凌日
（**injection–recovery**，天文界評估偵測門檻的標準做法），
把證據精準調到 Jeffreys 的決定邊界附近，再掃先驗寬度：

![Lindley 翻盤](../figures/06_weak_signal.png)

In [7]:
W = RES['weak']
print(f"注入深度掃描（真實 Kepler-10b 是 {0.0125**2*1e6:.0f} ppm，log10 B = "
      f"{RES['evidence']['kepler10b']['log10_B_M1M0']:.0f}）：")
for dep, b in zip(W['depth_ppm'], W['depth_log10B']):
    print(f"  深度 {dep:5.1f} ppm → log10 B(M1/M0) = {b:+7.2f}")
print(f"\n選 {W['chosen_depth_ppm']:.1f} ppm 作為邊緣案例，掃先驗寬度："
      f"（資料完全不變，只動先驗）")
for k, b in zip(W['ks'], W['k_log10B']):
    print(f"  k={k:>6.1f} → log10 B = {b:+7.2f}   "
          f"{'支持行星' if b > 0 else '支持純雜訊 ← 翻盤'}")

注入深度掃描（真實 Kepler-10b 是 156 ppm，log10 B = 167）：
  深度   3.6 ppm → log10 B(M1/M0) =   -1.15
  深度   4.4 ppm → log10 B(M1/M0) =   -0.62
  深度   5.3 ppm → log10 B(M1/M0) =   +0.35
  深度   6.2 ppm → log10 B(M1/M0) =   +1.74
  深度   7.8 ppm → log10 B(M1/M0) =   +4.71

選 5.3 ppm 作為邊緣案例，掃先驗寬度：（資料完全不變，只動先驗）
  k=   1.0 → log10 B =   +0.35   支持行星
  k=   3.0 → log10 B =   -0.36   支持純雜訊 ← 翻盤
  k=  10.0 → log10 B =   -0.68   支持純雜訊 ← 翻盤
  k=  30.0 → log10 B =   -0.86   支持純雜訊 ← 翻盤
  k= 100.0 → log10 B =   -0.52   支持純雜訊 ← 翻盤


**同一份資料、同一個似然函數，只把先驗放寬，判決就從「有行星」翻成「沒有」。**

這不是實作出錯，是貝氏因子的真實性質。實務上的應對：

1. **報告先驗，並報告敏感度**——不要只報一個 $B$ 值（這個專案的做法）。
2. **先驗要有物理依據**，不能為了「無資訊」而隨手放到很寬。
   「$R_p/R_*$ 上限 0.2」不是隨便選的，是行星的物理半徑上限。
   *在貝氏因子裡，「無資訊先驗」不是安全的預設值——它會系統性地懲罰複雜模型。*
3. 弱證據就老實說**證據弱**，不要靠先驗把它推過門檻。

## 5 · 限制與誠實的部分

- **M2 是唯象模型**：次食沿用主食的幾何（圓軌道假設），深度乘上表面亮度比 $J$。
  真正的食雙星建模（如 `ellc`、`PHOEBE`）還要處理橢圓軌道、潮汐變形、掩食/凌星幾何差異。
  對「有沒有次食」這個判別問題，這個簡化抓住了關鍵自由度，但不該拿去做雙星參數測定。
- **沒有建模相位曲線**（下一格有實測）。食雙星除了兩個食，還有**橢球變形**
  （伴星被潮汐拉長，產生週期 $P/2$ 的亮度調制）、反射與都卜勒增亮。
  這件事有兩層意義：(a) 它是食雙星的**第三個獨立證據**，而我們的判決完全沒用上它——
  結論其實比報告的更穩；(b) M1 與 M2 都假設平坦基線，這個未建模的調制被 jitter
  **對稱地**吸收，不偏袒任何一方，但確實讓次食深度帶有 ~100 ppm 等級的系統性偏差。
- **只測了一個 FALSE POSITIVE**。要宣稱這套流程「可靠」，該跑整批 KOI 並給出
  完整的混淆矩陣（真陽/假陽率）。一個成功案例證明方法可行，不證明它穩健。
- **模型空間不完整**。真實的 false positive 還有背景食雙星（BEB）、
  儀器假訊號、恆星黑子等。$p(M\mid D)$ 只在**列出來的**模型之間分配機率——
  贏家永遠只是「候選名單裡最好的」，不是「真相」。
- **次食深度與軌道離心率簡併**。這裡固定次食在相位 0.5；離心軌道會讓它偏移，
  真實分析要把 $e\cos\omega$ 放進來。
- $\log Z$ 的數值精度已用多 seed 量化（見第 2 節），
  但 Kepler-10b 的 M1/M2 差距本來就只有幾 nat——**這個結論是穩的，不是壓倒性的**。

In [8]:
# 實測 KOI-6017.01 的食外亮度：如果有橢球變形，相位 ±0.25 附近應該是峰、兩個食附近是谷
d = data.prepare('koi6017', DATA)
ph, fx = d['phase'], d['flux']
print('KOI-6017.01 食外亮度（相對基線，ppm）：')
for lo, hi in [(-0.45,-0.35), (-0.35,-0.25), (-0.25,-0.15), (-0.15,-0.08),
               (0.08,0.15), (0.15,0.25), (0.25,0.35), (0.35,0.45)]:
    m = (ph > lo) & (ph < hi)
    bar = '█' * max(0, int((np.mean(fx[m])-1)*1e6/20))
    print(f'  相位 [{lo:+.2f},{hi:+.2f}]: {(np.mean(fx[m])-1)*1e6:+8.1f} ppm  {bar}')
print('\n→ ±0.25~0.35 是峰、±0.08~0.15 是谷：週期 P/2 的橢球變形特徵。')
print('  食雙星的第三個證據，而我們的模型完全沒用上它。')

KOI-6017.01 食外亮度（相對基線，ppm）：
  相位 [-0.45,-0.35]:    +15.2 ppm  
  相位 [-0.35,-0.25]:    +93.1 ppm  ████
  相位 [-0.25,-0.15]:    +41.5 ppm  ██
  相位 [-0.15,-0.08]:   -112.7 ppm  
  相位 [+0.08,+0.15]:   -182.8 ppm  
  相位 [+0.15,+0.25]:   +208.1 ppm  ██████████
  相位 [+0.25,+0.35]:   +259.6 ppm  ████████████
  相位 [+0.35,+0.45]:    -97.5 ppm  

→ ±0.25~0.35 是峰、±0.08~0.15 是谷：週期 P/2 的橢球變形特徵。
  食雙星的第三個證據，而我們的模型完全沒用上它。
